<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/pacman/multi_experts_porow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# miso sans decay, thres factor

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
   "seeds": {1: 379229453, 2: 232504, 3: 5554205,
              42:590212, 43:682036, 44: 912403, 11:622344, 12: 742341},
    "weights":   {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "n_train":   200,
    "n_test":    5000,
    "n_levels":  0,       # nb de fonctions f_i cylindriques
    "n_levels_wnd": 0,    # nb de fonctions f_i Wendland anisotropes (support compact ->
                          # experts plus indépendants ; régularité C^{2k} minimale telle
                          # que 2k >= ordre max des poids Sobolev)
    "k_loss":    3.0,     # sélection finale : garder les experts avec loss <= k_loss * meilleure loss
    "deg_poly_expert": -1,      # degré de l'expert polynomial (dérivées analytiques exactes)
                               # -1 -> expert polynomial désactivé
    "n_dict":    5000,     # taille dictionnaire par niveau
    "n_centres": 800,      # centres retenus par niveau
    "sigma_min": 0.1,      # borne inférieure des sigma_i
    "sigma_max": 3.0,      # borne supérieure des sigma_i
    "lambda_reg": 1e-5,    # UNIQUE régularisation Sobolev (Phase 1, sélection, Phase 2) :
                           # la norme Sobolev est indépendante de la base -> une seule loss
    "thres_factor": 1,
    "n_G":       2000,      # points pour G de chaque f_i
    "n_G_H":     5000,     # points pour la Gram finale sur H
    "n_levels_miso": 25,     # niveaux multi-isotropes : UN sigma aléatoire par centre,
                            # log-uniforme dans [sigma_min, sigma_max] (0 -> étape sautée)
    "miso_greedy": True,    # True  : les niveaux miso sont construits en cascade gloutonne
                            #         (chaque niveau fitte le RÉSIDU du précédent) et seul
                            #         le DERNIER (= la somme cumulée) est conservé comme expert.
                            # False : anciens niveaux miso indépendants (tous conservés).
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    norm_c = np.sqrt(np.mean(coeffs**2))
    if norm_c > 0:
        nc = coeffs / norm_c

        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0:
                        term *= x[:, j]**e
                y += c * term
            return y

        return Pn
    return P
#P1, indices1, coeffs1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][1])
#P2, indices2, coeffs2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][2])
#P3, indices3, coeffs3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][3])

#P1 = normalize_polynomial(P1, indices1, coeffs1)
#P2 = normalize_polynomial(P2, indices2, coeffs2)
#P3 = normalize_polynomial(P3, indices3, coeffs3)


def P1(X):
    x1, x2, x3, x4 = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return x1**2 + x2**2 + x3**2 - x4**2

def P2(X):
    x1, x2, x3, x4 = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return x1**2 + x2**2 + x3**2 - 0.5*x4**2

def P3(X):
    x1, x2, x3, x4 = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    return x1 + x2 + 2*x3 - 2*x4

def Q(X):
    return P1(X) * P2(X) * P3(X)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps=1e-5; d=X.shape[1]; p1=P1(X); p2=P2(X); p3=P3(X)
    grad=np.zeros_like(X)
    for k in range(d):
        Xp=X.copy(); Xp[:,k]+=eps; Xm=X.copy(); Xm[:,k]-=eps
        dp1=(P1(Xp)-P1(Xm))/(2*eps); dp2=(P2(Xp)-P2(Xm))/(2*eps); dp3=(P3(Xp)-P3(Xm))/(2*eps)
        grad[:,k]=dp1*p2*p3+p1*dp2*p3+p1*p2*dp3
    return grad

# ── CORRECTIF 1 : projection de Gauss-Newton sur {Q=0} ────────────────────────
# L'ancienne version faisait une descente de gradient normalisée sur Q^2, avec
# un pas de longueur FIXE (lr=0.05) indépendant de la distance à la variété.
# Près de {Q=0} le gradient s'aplatit et le pas normalisé oscille sans jamais
# passer sous tol -> taux de convergence très variable selon le tirage de Q
# (observé : de 4% à 90%+ selon les graines), voire blocage.
# Le pas de Newton x <- x - Q*gradQ/||gradQ||^2 annule Q au premier ordre :
# convergence quadratique, taux >85% sur tous les tirages testés, 0 NaN.
def project_to_Q_zero(X_init, n_steps=60, tol=1e-4, damp=1.0):
    X = X_init.copy()
    for _ in range(n_steps):
        q  = Q(X)
        gq = grad_Q_analytical(X)
        g2 = np.sum(gq*gq, axis=1, keepdims=True) + 1e-12
        X  = X - damp * (q[:, None] * gq) / g2
        X  = np.clip(X, 0., 1.)
        if np.abs(Q(X)).max() < tol * 0.1:
            break
    return X, np.abs(Q(X)) < tol

# ── CORRECTIF 2 : échantillonnage borné + filtre NaN + taux cumulé ────────────
# L'ancienne version bouclait sans limite (risque de blocage silencieux si un
# tirage de Q est mal conditionné) et affichait le taux du DERNIER batch, pas
# le taux global (trompeur quand il ne reste que peu de points à collecter).
def sample_on_Q_zero(n_target, d, seed=None, max_batches=200):
    rng = np.random.default_rng(seed)
    collected = []; n_col = 0; n_seen = 0; n_ok = 0
    for _ in range(max_batches):
        if n_col >= n_target:
            break
        n_batch = min(max((n_target - n_col) * 4, 200), 20000)
        Xi = rng.uniform(0, 1, (int(n_batch), d))
        Xp, conv = project_to_Q_zero(Xi)
        good = Xp[conv]
        good = good[np.all(np.isfinite(good), axis=1)]   # ceinture de sécurité anti-NaN
        n_seen += len(Xi); n_ok += int(conv.sum())
        if len(good) > 0:
            collected.append(good); n_col += len(good)
        print(f"  collectés:{n_col}/{n_target} (cumulé {n_ok/max(n_seen,1):.1%})", end='\r')
    print()
    if n_col < n_target:
        raise RuntimeError(
            f"sample_on_Q_zero: seulement {n_col}/{n_target} points obtenus après "
            f"{max_batches} batchs (taux cumulé {n_ok/max(n_seen,1):.1%}). "
            f"La variété {{Q=0}} est probablement mal conditionnée pour ce tirage "
            f"de seeds[1,2,3] -- essayez d'autres graines.")
    return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d=params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],    d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],     d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"],d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(
    params["n_ambiant"], 4, params["n_terms_poly"],
    seed=params["seeds"][11]
)

P_target2, indicest2, coeffst2 = random_sparse_polynomial(
    params["n_ambiant"], 4, params["n_terms_poly"],
    seed=params["seeds"][12]
)

Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest2, coeffst2)

def target_function(X):
    return np.minimum(np.abs(Ptarget1(X)),1) +  np.minimum(np.abs(Ptarget2(X)),1)
def polynomial_to_string(indices, coeffs):
    terms = []

    for c, alpha in zip(coeffs, indices):
        if abs(c) < 1e-12:
            continue

        monomial = []
        for j, e in enumerate(alpha):
            if e > 0:
                monomial.append(f"x{j+1}" if e == 1 else f"x{j+1}^{e}")

        if monomial:
            term = " ".join(monomial)
        else:
            term = "1"

        terms.append(f"({c:+.4f})*{term}")

    return " ".join(terms)


# Coefficients normalisés RMS
norm1 = np.sqrt(np.mean(coeffst1**2))
norm2 = np.sqrt(np.mean(coeffst2**2))

print("\nP_target1 =")
print(polynomial_to_string(indicest1, coeffst1 / norm1))

print("\nP_target2 =")
print(polynomial_to_string(indicest2, coeffst2 / norm2))
y_train=target_function(X_train); y_test=target_function(X_test)
X_all=np.vstack([X_train,X_unlabeled]); N_all=len(X_all)

# Alerte : si la cible est saturée (les deux termes valent 1) sur une grosse
# fraction du test, la classification par seuil médian peut dégénérer.
_sat = np.mean(y_test >= 1.999)
if _sat > 0.3:
    print(f"  [!] cible saturée sur {_sat:.1%} du test -- risque de classe unique "
          f"au seuillage médian")

# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES CYLINDRIQUES
# ══════════════════════════════════════════════════════════════════════════════
def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=(len(centers),d))
    sigmas=np.exp(log_s)
    return centers, sigmas

def sample_candidates_miso(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    """Multi-isotrope (ancienne famille iso) : gaussiennes isotropes avec UN sigma
    aléatoire par centre, log-uniforme dans [sigma_min, sigma_max], répété sur
    les d dimensions."""
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=len(centers))                      # UN sigma par centre
    sigmas=np.tile(np.exp(log_s)[:,None],(1,d))               # répété sur les d dims
    return centers, sigmas

def cyl_features(X, centers, sigmas):
    diff=X[:,None,:]-centers[None,:,:]
    sq_w=np.sum(diff**2/sigmas[None,:,:]**2, axis=2)
    return np.exp(-sq_w)

def cyl_derivatives(X, centers, sigmas, weights=None):
    d_=X.shape[1]; n=X.shape[0]; m=centers.shape[0]
    diff=X[:,None,:]-centers[None,:,:]
    inv_s2=1./sigmas**2
    diff_w=diff*inv_s2[None,:,:]
    sq_w=np.sum(diff*diff_w, axis=2)
    phi=np.exp(-sq_w)
    grad=-2*diff_w*phi[:,:,None]
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id=np.eye(d_)
        diag=-2*inv_s2[None,:,:,None]*Id[None,None,:,:]
        cross=4*np.einsum('nmi,nmj->nmij',diff_w,diff_w)
        hess=(diag+cross)*phi[:,:,None,None]
    else: hess=None
    if need_third:
        Id=np.eye(d_)
        inv_s2_bc=inv_s2[None,:,:]
        cubic=-8*np.einsum('nmi,nmj,nmk->nmijk',diff_w,diff_w,diff_w)
        sym=4*(np.einsum('ij,nmi,nmk->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('ik,nmi,nmj->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('jk,nmj,nmi->nmijk',Id,inv_s2_bc,diff_w))
        third=(cubic+sym)*phi[:,:,None,None,None]
    else: third=None
    return phi, grad, hess, third

# ── Wendland anisotropes ──────────────────────────────────────────────────────
def _wnd_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    sd = lambda num, r: np.where(r > 1e-12, num/np.maximum(r, 1e-12), 0.0)
    if k == 1:      # C² : suffit si ordre max <= 2
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*rp**3,
                lambda r,rp: sd(60*rp**2, r),
                None)
    elif k == 2:    # C⁴ : suffit si ordre max <= 4
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -(56/3)*rp**5*(5*r+1),
                lambda r,rp: 560.*rp**4,
                lambda r,rp: sd(-2240.*rp**3, r))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 528*rp**6*(6*r+1),
                lambda r,rp: -22176.*rp**5)

_wnd_max_order = max([o for o, w in params["weights"].items() if w != 0])
WND_PHI, WND_H1, WND_H2, WND_H3 = _wnd_profile(_wnd_max_order)

def wnd_features(X, centers, sigmas):
    u = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def wnd_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    u  = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r  = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    inv_s  = 1./sigmas
    inv_s2 = inv_s**2
    us  = u*inv_s[None,:,:]
    phi = WND_PHI(r, rp)
    H1  = WND_H1(r, rp)
    grad = H1[:,:,None]*us
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        H2 = WND_H2(r, rp)
        hess = (H2[:,:,None,None]*np.einsum('nmi,nmj->nmij', us, us)
                + H1[:,:,None,None]*(Id[None,None,:,:]*inv_s2[None,:,:,None]))
    else: hess=None
    if need_third:
        assert WND_H3 is not None, "poids d'ordre 3 : profil Wendland C⁴ minimum requis"
        Id = np.eye(d_)
        H3 = WND_H3(r, rp)
        cubic = H3[:,:,None,None,None]*np.einsum('nmi,nmj,nmk->nmijk', us, us, us)
        sym = (np.einsum('ij,mi,nmk->nmijk', Id, inv_s2, us)
              +np.einsum('ik,mi,nmj->nmijk', Id, inv_s2, us)
              +np.einsum('jk,mj,nmi->nmijk', Id, inv_s2, us))
        third = cubic + WND_H2(r, rp)[:,:,None,None,None]*sym
    else: third=None
    return phi, grad, hess, third

def build_G(phi, grad, hess, third, weights, n):
    G=np.zeros((phi.shape[1],phi.shape[1]))
    w0=weights.get(0,0.)
    if w0: G+=w0*(phi.T@phi)/n
    w1=weights.get(1,0.)
    if w1 and grad  is not None: G+=w1*np.einsum('xik,xjk->ij',    grad, grad, optimize='optimal')/n
    w2=weights.get(2,0.)
    if w2 and hess  is not None: G+=w2*np.einsum('xikl,xjkl->ij',  hess, hess, optimize='optimal')/n
    w3=weights.get(3,0.)
    if w3 and third is not None: G+=w3*np.einsum('xiklm,xjklm->ij',third,third,optimize='optimal')/n
    return G

def sobolev_norm2(derivs, weights, idx=None):
    tot = 0.
    for key, order, axes in (('phi',0,()), ('grad',1,(1,)),
                             ('hess',2,(1,2)), ('third',3,(1,2,3))):
        w = weights.get(order, 0.)
        if not w:            continue
        T = derivs.get(key)
        if T is None:        continue
        if idx is not None:  T = T[idx]
        tot += w * (np.mean(T**2) if order == 0
                    else np.mean(np.sum(T**2, axis=axes)))
    return tot

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : calculer n_levels fonctions f_i indépendantes
# ══════════════════════════════════════════════════════════════════════════════
MISO_GREEDY = bool(params.get("miso_greedy", False)) and params["n_levels_miso"] > 0

level_specs = ([("cyl",  i) for i in range(params["n_levels"])] +
               [("miso", i) for i in range(params["n_levels_miso"])] +
               [("wnd",  i) for i in range(params["n_levels_wnd"])])

n_experts_total = len(level_specs) - (params["n_levels_miso"] - 1 if MISO_GREEDY else 0)
n_levels_total  = n_experts_total

expert_names = []
for typ, i in level_specs:
    if typ == "miso" and MISO_GREEDY:
        if i == params["n_levels_miso"] - 1:
            expert_names.append(f"misoG{params['n_levels_miso']}")
        continue
    expert_names.append(f"{typ}_{i+1:02d}")

print(f"\nPhase 1 : calcul de {n_experts_total} experts "
      f"({params['n_levels']} cyl + "
      f"{params['n_levels_miso']} miso"
      f"{' [glouton -> 1 expert]' if MISO_GREEDY else ''} + "
      f"{params['n_levels_wnd']} wnd C{2*max(1,int(np.ceil(_wnd_max_order/2)))})...")

F_train = np.zeros((len(X_train), n_experts_total))
F_test  = np.zeros((len(X_test),  n_experts_total))
F_all   = np.zeros((N_all,        n_experts_total))

fi_derivs_list = []
times_p1 = []

gr_res     = y_train.copy()
gr_train   = np.zeros(len(X_train))
gr_test    = np.zeros(len(X_test))
gr_all     = np.zeros(N_all)
gr_derivs  = None
gr_h2_hist = []

slot = 0
for level, (lev_type, lev_idx) in enumerate(level_specs):
    t0 = time.time()
    greedy_step = (lev_type == "miso") and MISO_GREEDY
    last_greedy = greedy_step and (lev_idx == params["n_levels_miso"] - 1)
    y_fit = gr_res if greedy_step else y_train

    rng = np.random.default_rng(
        seed={"cyl":0, "wnd":2000, "miso":3000}[lev_type]+lev_idx)

    feat_fn  = wnd_features    if lev_type=="wnd" else cyl_features
    deriv_fn = wnd_derivatives if lev_type=="wnd" else cyl_derivatives
    if lev_type == "miso":
        candidates, sigmas_cand = sample_candidates_miso(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)
    else:
        candidates, sigmas_cand = sample_candidates(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)

    A_cand = feat_fn(X_train, candidates, sigmas_cand)
    corr   = (A_cand.T @ y_fit) / len(X_train)
    scores = corr**2

    k     = min(params["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers_sel = candidates[top_k]
    sigmas_sel  = sigmas_cand[top_k]

    A     = A_cand[:, top_k]
    Atest = feat_fn(X_test, centers_sel, sigmas_sel)
    Aall  = feat_fn(X_all,  centers_sel, sigmas_sel)

    phi, grad, hess, third = deriv_fn(
        X_all, centers_sel, sigmas_sel, params["weights"])

    n_G   = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G(phi[idx_G],
                grad[idx_G]  if grad  is not None else None,
                hess[idx_G]  if hess  is not None else None,
                third[idx_G] if third is not None else None,
                params["weights"], n_G)

    lambda_reg = params["lambda_reg"]
    n = A.shape[0]
    M   = (A.T@A)/n + lambda_reg*G
    rhs = (A.T@y_fit)/n

    eigvals,eigvecs = np.linalg.eigh(M)
    thresh = max(lambda_reg*params["thres_factor"], 0); mask = eigvals > thresh
    V=eigvecs[:,mask]; S=eigvals[mask]; coeffs=V@((V.T@rhs)/S)

    f_tr = A     @ coeffs
    f_te = Atest @ coeffs
    f_al = Aall  @ coeffs

    step_derivs = {
        'phi':   phi   @ coeffs,
        'grad':  np.einsum('xjk,j->xk',   grad, coeffs) if grad  is not None else None,
        'hess':  np.einsum('xjkl,j->xkl', hess, coeffs) if hess  is not None else None,
        'third': np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None,
    }

    loss_data  = np.mean((y_fit - f_tr)**2)
    loss_reg   = lambda_reg * coeffs @ G @ coeffs
    loss_total = loss_data + loss_reg
    t1=time.time(); times_p1.append(t1-t0)

    if greedy_step:
        gr_train += f_tr; gr_test += f_te; gr_all += f_al
        gr_res    = y_train - gr_train
        if gr_derivs is None:
            # copie explicite : évite l'aliasing avec step_derivs
            gr_derivs = {k: (None if v is None else v.copy()) for k, v in step_derivs.items()}
        else:
            for key in ('phi','grad','hess','third'):
                if step_derivs[key] is None or gr_derivs[key] is None:
                    gr_derivs[key] = None
                else:
                    gr_derivs[key] = gr_derivs[key] + step_derivs[key]

        mse_cum = mean_squared_error(y_test, gr_test)
        h2_step = sobolev_norm2(step_derivs, params["weights"])
        h2_cum  = sobolev_norm2(gr_derivs,   params["weights"])
        gr_h2_hist.append(h2_cum)
        dh2 = h2_cum - (gr_h2_hist[-2] if len(gr_h2_hist) > 1 else 0.)
        print(f"  [glouton {lev_idx+1}/{params['n_levels_miso']}] miso | "
              f"rang={mask.sum():3d}/{k} | loss_res={loss_total:.6f} "
              f"(data={loss_data:.6f} reg={loss_reg:.6f}) | "
              f"MSE_cum={mse_cum:.6f} | résidu_train={np.mean(gr_res**2):.6f} | "
              f"{times_p1[-1]:.1f}s\n"
              f"      └─ ||f||_H : étage={np.sqrt(h2_step):.4f}  "
              f"CUMULÉ={np.sqrt(h2_cum):.4f}  (||.||²_cum={h2_cum:.4e}, "
              f"Δ||.||²={dh2:+.3e}) | "
              f"loss_H_cum={np.mean(gr_res**2) + lambda_reg*h2_cum:.6f}")

        if last_greedy:
            F_train[:,slot] = gr_train
            F_test [:,slot] = gr_test
            F_all  [:,slot] = gr_all
            fi_derivs_list.append(gr_derivs)
            print(f"  {expert_names[slot]:7s} | cascade conservée | "
                  f"MSE={mean_squared_error(y_test, gr_test):.6f} | "
                  f"||f||_H={np.sqrt(gr_h2_hist[-1]):.4f}")
            print("      trajectoire ||f_cum||_H : "
                  + " -> ".join(f"{np.sqrt(v):.4f}" for v in gr_h2_hist))
            slot += 1
        continue

    F_train[:,slot] = f_tr
    F_test [:,slot] = f_te
    F_all  [:,slot] = f_al
    fi_derivs_list.append(step_derivs)

    mse_i = mean_squared_error(y_test, f_te)
    print(f"  {expert_names[slot]:7s} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_total:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"MSE={mse_i:.6f} | σ∈[{sigmas_sel.min():.2f},{sigmas_sel.max():.2f}] | "
          f"{times_p1[-1]:.1f}s")
    slot += 1


# ══════════════════════════════════════════════════════════════════════════════
# AJOUT DES EXPERTS RBF ET RIDGE AVEC DÉRIVÉES SOBOLEV SANS TENSEUR COMPLET
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import PolynomialFeatures

USE_POLY_EXPERT = params["deg_poly_expert"] >= 0
print("\nCalibration expert polynomial..." if USE_POLY_EXPERT
      else "\nExpert polynomial DÉSACTIVÉ (deg_poly_expert < 0)")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
n_H   = len(idx_H)
X_H   = X_all[idx_H]

w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.)
w3 = params["weights"].get(3, 0.)

if USE_POLY_EXPERT:
    deg_e   = params["deg_poly_expert"]
    d_      = params["n_ambiant"]
    poly_e  = PolynomialFeatures(degree=deg_e, include_bias=False)
    Phi_tr  = poly_e.fit_transform(X_train)
    ridge_e = Ridge(alpha=1e-8)
    ridge_e.fit(Phi_tr, y_train)
    powers_e    = poly_e.powers_
    coefs_e     = ridge_e.coef_.copy()
    intercept_e = float(ridge_e.intercept_)

    pred_poly_train = ridge_e.predict(Phi_tr)
    pred_poly_test  = ridge_e.predict(poly_e.transform(X_test))

    def poly_deriv_eval(X, powers, coefs, dims=()):
        P = powers.astype(np.float64).copy()
        mult = coefs.astype(np.float64).copy()
        for m in dims:
            mult = mult * P[:, m]
            P[:, m] -= 1
        valid = mult != 0
        if not np.any(valid):
            return np.zeros(len(X))
        Xp = X[:, None, :] ** P[None, valid, :]
        return Xp.prod(axis=2) @ mult[valid]

    delta_phi_poly = poly_deriv_eval(X_H, powers_e, coefs_e) + intercept_e

    if w1 or w2 or w3:
        delta_grad_poly = np.stack(
            [poly_deriv_eval(X_H, powers_e, coefs_e, (k_,)) for k_ in range(d_)], axis=1)
    else:
        delta_grad_poly = None

    if w2 or w3:
        delta_hess_poly = np.zeros((n_H, d_, d_))
        for k_ in range(d_):
            for l_ in range(k_, d_):
                v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_))
                delta_hess_poly[:,k_,l_] = v; delta_hess_poly[:,l_,k_] = v
    else:
        delta_hess_poly = None

    if w3:
        delta_third_poly = np.zeros((n_H, d_, d_, d_))
        from itertools import permutations
        for k_ in range(d_):
            for l_ in range(k_, d_):
                for m_ in range(l_, d_):
                    v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_, m_))
                    for a,b,c in set(permutations((k_,l_,m_))):
                        delta_third_poly[:,a,b,c] = v
    else:
        delta_third_poly = None
    norm_poly = 0.
    if w1: norm_poly += w1*np.mean(np.sum(delta_grad_poly**2,  axis=1))
    if w2: norm_poly += w2*np.mean(np.sum(delta_hess_poly**2,  axis=(1,2)))
    if w3: norm_poly += w3*np.mean(np.sum(delta_third_poly**2, axis=(1,2,3)))
    loss_data_p = np.mean((y_train - pred_poly_train)**2)
    print(f"  Poly{deg_e}  | rang={Phi_tr.shape[1]:3d}/{Phi_tr.shape[1]} | "
          f"loss={loss_data_p + params['lambda_reg']*norm_poly:.6f} "
          f"(data={loss_data_p:.6f} reg={params['lambda_reg']*norm_poly:.6f}) | "
          f"MSE={mean_squared_error(y_test, pred_poly_test):.6f}")

    F_train = np.column_stack([F_train, pred_poly_train])
    F_test  = np.column_stack([F_test,  pred_poly_test])

    fi_derivs_list.append({
        'phi': delta_phi_poly,   'grad': delta_grad_poly,
        'hess': delta_hess_poly, 'third': delta_third_poly})

    expert_names += [f"Poly{deg_e}"]

print(f"  H étendu à {len(fi_derivs_list)} fonctions")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : argmin sur H = <f_1,...,f_n_levels>
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram finale sur H ({len(fi_derivs_list)} fonctions, n_G_H={n_H})...")

n_lev = len(fi_derivs_list)
G_H = np.zeros((n_lev, n_lev))
w0=params["weights"].get(0,0.); w1=params["weights"].get(1,0.)
w2=params["weights"].get(2,0.); w3=params["weights"].get(3,0.)

if w0:
    PHI_H = np.stack([
        fi['phi'] if fi['phi'].shape[0] == n_H else fi['phi'][idx_H]
        for fi in fi_derivs_list], axis=1)
    G_H += w0*(PHI_H.T@PHI_H)/n_H

if w1:
    idx_with_grad = [i for i, fi in enumerate(fi_derivs_list) if fi['grad'] is not None]
    if idx_with_grad:
        GRAD_H = np.stack([
            (fi_derivs_list[i]['grad'] if fi_derivs_list[i]['grad'].shape[0] == n_H
             else fi_derivs_list[i]['grad'][idx_H])
            for i in idx_with_grad], axis=1)
        G_part = np.einsum('xik,xjk->ij', GRAD_H, GRAD_H, optimize='optimal') * w1 / n_H
        G_H[np.ix_(idx_with_grad, idx_with_grad)] += G_part

if w2:
    idx_with_hess = [i for i, fi in enumerate(fi_derivs_list) if fi['hess'] is not None]
    if idx_with_hess:
        HESS_H = np.stack([
            (fi_derivs_list[i]['hess'] if fi_derivs_list[i]['hess'].shape[0] == n_H
             else fi_derivs_list[i]['hess'][idx_H])
            for i in idx_with_hess], axis=1)
        G_part = np.einsum('xikl,xjkl->ij', HESS_H, HESS_H, optimize='optimal') * w2 / n_H
        G_H[np.ix_(idx_with_hess, idx_with_hess)] += G_part

if w3:
    idx_with_third = [i for i, fi in enumerate(fi_derivs_list) if fi['third'] is not None]
    if idx_with_third:
        thirds = []
        for i in idx_with_third:
            t = fi_derivs_list[i]['third']
            thirds.append(t if t.shape[0] == n_H else t[idx_H])
        THIRD_H = np.stack(thirds, axis=1)
        G_partial = np.einsum('xiklm,xjklm->ij', THIRD_H, THIRD_H, optimize='optimal') * w3 / n_H
        for ii, i in enumerate(idx_with_third):
            for jj, j in enumerate(idx_with_third):
                G_H[i, j] += G_partial[ii, jj]

print(f"  G_H calculée ({n_H} pts)")

# ── SÉLECTION DES EXPERTS PAR LOSS ────────────────────────────────────────────
n_loc  = len(y_train)
m_vec  = F_train.T @ y_train / n_loc
q_vec  = np.sum(F_train**2, axis=0) / n_loc
denom  = q_vec + params["lambda_reg"] * np.diag(G_H)

# ── CORRECTIF 3 : garde contre les experts dégénérés ──────────────────────────
# Si un expert est ~nul sur le train (ex: centre Wendland à support compact
# dont aucun centre ne couvre les n_train points, plus probable quand n_train
# est petit), denom peut valoir 0 -> 0/0 = nan qui contamine tout le reste
# (best_loss, sel, alpha...). On neutralise ces experts au lieu de planter.
bad_denom = denom <= 1e-14
if bad_denom.any():
    print(f"  [!] {bad_denom.sum()} expert(s) dégénéré(s) sur le train "
          f"(denom~0) : {[expert_names[i] for i in np.where(bad_denom)[0]]} -- écartés")
denom_safe = np.where(bad_denom, 1.0, denom)
c_opt  = np.where(bad_denom, 0.0, m_vec / denom_safe)
expert_losses = np.mean(y_train**2) - m_vec**2 / denom_safe
expert_losses = np.where(bad_denom, np.inf, expert_losses)

best_loss   = expert_losses.min()
sel         = expert_losses <= params["k_loss"] * best_loss
sel_idx     = np.where(sel)[0]

print(f"\n  Sélection experts (k_loss={params['k_loss']}, "
      f"seuil={params['k_loss']*best_loss:.3e}) :")
for i in range(n_lev):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f"    {expert_names[i]:7s} : loss={expert_losses[i]:.3e}  c*={c_opt[i]:+.3f}  [{tag}]")
print(f"  -> {sel.sum()}/{n_lev} experts retenus")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:,  sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]

n_tr = F_train_sel.shape[0]
M_H  = (F_train_sel.T@F_train_sel)/n_tr + params["lambda_reg"]*G_H_sel
rhs_H = (F_train_sel.T@y_train)/n_tr

eigvals_H,eigvecs_H = np.linalg.eigh(M_H)
thresh_H = max(params["lambda_reg"], 1e-10)
mask_H   = eigvals_H > thresh_H
V_H=eigvecs_H[:,mask_H]; S_H=eigvals_H[mask_H]
alpha_sel = V_H@((V_H.T@rhs_H)/S_H)

alpha = np.zeros(n_lev); alpha[sel_idx] = alpha_sel

pred_H_train = F_train_sel @ alpha_sel
pred_H_test  = F_test_sel  @ alpha_sel
mse_H = mean_squared_error(y_test, pred_H_test)
mse_H_train = mean_squared_error(y_train, pred_H_train)

print(f"  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_lev} experts)")
print(f"  MSE train = {mse_H_train:.6f}  MSE test = {mse_H:.6f}")
print(f"  alpha = {alpha.round(3)}")


# ══════════════════════════════════════════════════════════════════════════════
# ABLATION : que vaut H SANS l'expert glouton ?
# On rejoue exactement la même Phase 2 (même k_loss, même λ, même solveur) sur
# le sous-ensemble d'experts privé de misoG, et on compare.
#
# ATTENTION AU RÉSULTAT : avec n_train petit (<=30 environ) et plusieurs
# experts qui interpolent quasi parfaitement le train (résidu ~1e-5), F_train
# est numériquement de rang très bas : les experts sont colinéaires SUR LE
# TRAIN (corr_L2~1.000), et n'importe quelle combinaison qui reproduit y sur
# les n_train points convient -- c'est le régularisateur Sobolev qui
# départage. Le "gain" mesuré ici peut alors refléter ce quasi-hasard plutôt
# qu'une vraie contribution du glouton. Ne pas le croire sur un run unique ;
# répliquer sur plusieurs tirages de graines avant de conclure.
# ══════════════════════════════════════════════════════════════════════════════
def solve_H(keep_mask):
    """Phase 2 restreinte aux experts de keep_mask (bool, taille n_lev).
    Refait la sélection k_loss À L'INTÉRIEUR de ce sous-ensemble, puis résout.
    Retourne (mse_test, mse_train, sel_locale, alpha_complet, rang)."""
    kidx = np.where(keep_mask)[0]
    if len(kidx) == 0:
        return np.nan, np.nan, np.zeros(n_lev, bool), np.zeros(n_lev), 0

    losses_k = expert_losses[kidx]
    finite_k = np.isfinite(losses_k)
    if not finite_k.any():
        return np.nan, np.nan, np.zeros(n_lev, bool), np.zeros(n_lev), 0
    sel_k = finite_k & (losses_k <= params["k_loss"] * losses_k[finite_k].min())
    sidx  = kidx[sel_k]

    Ftr = F_train[:, sidx]; Fte = F_test[:, sidx]
    GH  = G_H[np.ix_(sidx, sidx)]

    n_t = Ftr.shape[0]
    M   = (Ftr.T @ Ftr) / n_t + params["lambda_reg"] * GH
    rhs = (Ftr.T @ y_train) / n_t
    ev, evec = np.linalg.eigh(M)
    mk = ev > max(params["lambda_reg"], 1e-10)
    V = evec[:, mk]; S = ev[mk]
    a_s = V @ ((V.T @ rhs) / S)

    a_full = np.zeros(n_lev); a_full[sidx] = a_s
    sel_full = np.zeros(n_lev, bool); sel_full[sidx] = True
    return (mean_squared_error(y_test,  Fte @ a_s),
            mean_squared_error(y_train, Ftr @ a_s),
            sel_full, a_full, int(mk.sum()))


greedy_idx = [i for i, nm in enumerate(expert_names) if nm.startswith("misoG")]

if MISO_GREEDY and greedy_idx:
    g = greedy_idx[0]
    print("\n" + "="*70)
    print("ABLATION : H avec vs sans l'expert glouton")
    print("="*70)

    mask_with    = np.ones(n_lev, bool)
    mask_without = np.ones(n_lev, bool); mask_without[g] = False

    mse_w,  mse_w_tr,  sel_w,  a_w,  rk_w  = solve_H(mask_with)
    mse_wo, mse_wo_tr, sel_wo, a_wo, rk_wo = solve_H(mask_without)

    gain     = mse_wo - mse_w
    gain_rel = gain / mse_wo * 100 if mse_wo > 0 else np.nan

    print(f"  H avec  {expert_names[g]:8s} : MSE_test={mse_w:.6f}  "
          f"MSE_train={mse_w_tr:.6f}  rang={rk_w}/{sel_w.sum()}  "
          f"α_glouton={a_w[g]:+.4f}")
    print(f"  H sans  {expert_names[g]:8s} : MSE_test={mse_wo:.6f}  "
          f"MSE_train={mse_wo_tr:.6f}  rang={rk_wo}/{sel_wo.sum()}")
    verdict = ("AIDE" if gain > 0 else "NUIT" if gain < 0 else "neutre")
    print(f"  -> gain = {gain:+.6f} ({gain_rel:+.2f}%)   [{verdict}]  "
          f"[résultat sur UN SEUL tirage -- cf. avertissement ci-dessus]")

    mse_g_solo = mean_squared_error(y_test, F_test[:, g])
    #mse_best_other = min(mean_squared_error(y_test, F_test[:, i])
     #                    for i in range(n_lev) if i != g)
    print(f"\n  {expert_names[g]} seul : MSE={mse_g_solo:.6f} | "
          f"||f||_H={np.sqrt(G_H[g,g]):.4f} | loss={expert_losses[g]:.3e}")
    #print(f"  meilleur autre expert seul : MSE={mse_best_other:.6f}")
    #print(f"  H complet : MSE={mse_w:.6f}  "
    #      f"({'MIEUX' if mse_w < min(mse_g_solo, mse_best_other) else 'pas mieux'} "
     #     f"que tout expert seul -> complémentarité)")

    print(f"\n  Redondance de {expert_names[g]} (|corr| ; 1 = redondant) :")
    fg = F_train[:, g]
    for i in range(n_lev):
        if i == g: continue
        num_l2 = abs(fg @ F_train[:, i])
        den_l2 = np.linalg.norm(fg) * np.linalg.norm(F_train[:, i]) + 1e-30
        c_h = abs(G_H[g, i]) / (np.sqrt(G_H[g, g] * G_H[i, i]) + 1e-30)
        print(f"    vs {expert_names[i]:8s} : corr_L2={num_l2/den_l2:.3f}   "
              f"corr_H={c_h:.3f}")
    if np.all(np.array([abs(fg @ F_train[:, i]) /
              (np.linalg.norm(fg)*np.linalg.norm(F_train[:, i])+1e-30)
              for i in range(n_lev) if i != g]) > 0.99):
        print(f"\n  [!] corr_L2 > 0.99 avec TOUS les autres experts : sur le train, "
              f"{expert_names[g]} est quasi colinéaire au reste. Avec n_train="
              f"{len(y_train)}, ceci indique un système sous-déterminé -- le "
              f"'gain' ci-dessus est peu fiable en l'état (voir avertissement).")


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF (baseline de comparaison uniquement, hors H)...")
gamma_grid = np.logspace(-3,2,20)
param_grid = {'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv = GridSearchCV(KernelRidge(), param_grid=param_grid, cv=5,
                     scoring='neg_mean_squared_error', n_jobs=-1)
kr_cv.fit(X_train, y_train)
best_gamma = kr_cv.best_params_['gamma']
best_alpha = kr_cv.best_params_['alpha']
best_err_rbf = mean_squared_error(y_test, kr_cv.predict(X_test))
results = kr_cv.cv_results_; order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]; g = results['param_gamma'][idx]; a = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(g), alpha=float(a))
    kr.fit(X_train, y_train)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} TEST={mean_squared_error(y_test, kr.predict(X_test)):.6f}")

poly=PolynomialFeatures(degree=min(params["deg_P"],8),include_bias=False)
ridge_poly=Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train),y_train)
err_poly=mean_squared_error(y_test,ridge_poly.predict(poly.transform(X_test)))

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*70); print("RÉSUMÉ"); print("="*70)
print(f"Polynomial Ridge  : {err_poly:.6f}")
print(f"RBF (γ={best_gamma:.4f})  : {best_err_rbf:.6f}")
print(f"\nMSE individuelles des experts :")
for i in range(n_lev):
    mse_i = mean_squared_error(y_test, F_test[:,i])
    tag = "" if sel[i] else "  (écarté)"
    print(f"  {expert_names[i]:7s} : {mse_i:.6f}  loss={expert_losses[i]:.3e}{tag}")
print(f"\nSobolev sous-espace H : {mse_H:.6f}  (rang={mask_H.sum()})")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']} (unique)")
print(f"n_levels={params['n_levels']} (+{params['n_levels_miso']} miso"
      f"{' glouton->1' if MISO_GREEDY else ''}, "
      f"+{params['n_levels_wnd']} wnd, "
      f"poly={'OFF' if not USE_POLY_EXPERT else params['deg_poly_expert']})  "
      f"n_centres={params['n_centres']}  n_dict={params['n_dict']}  "
      f"k_loss={params['k_loss']} ({sel.sum()}/{n_lev} experts retenus)")
print(f"n_G={params['n_G']}  n_G_H={params['n_G_H']}")
print(f"σ∈[{params['sigma_min']},{params['sigma_max']}]")
print(f"temps phase 1 : {sum(times_p1):.1f}s total  ({np.mean(times_p1):.1f}s/niveau)")

from sklearn.metrics import roc_auc_score, accuracy_score

# ── CORRECTIF 4 : split par rang, pas par seuil médian brut ──────────────────
# target_function sature (plafonne) sur une grosse fraction du domaine selon
# le tirage des polynômes-cible (seeds[11,12]). Quand >50% des points sont au
# plafond, la médiane = le plafond, et (y > médiane) est structurellement vide
# -> une seule classe -> AUC/accuracy indéfinis. Un split par rang avec bris
# d'égalité aléatoire garantit toujours deux classes non vides, quel que soit
# le nombre de valeurs à égalité au seuil.
def rank_split(y, seed=0):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(y))
    order = perm[np.argsort(y[perm], kind='stable')]
    labels = np.zeros(len(y), dtype=int)
    labels[order[len(order)//2:]] = 1
    return labels

seuil = np.median(y_test)
_frac_sat = np.mean(y_test >= y_test.max() - 1e-9)
if _frac_sat > 0.5:
    print(f"  [!] {_frac_sat:.1%} de y_test au plafond -- seuillage médian dégénère "
          f"(médiane = plafond) -> split par rang utilisé à la place")
y_test_class  = rank_split(y_test,  seed=1)
y_train_class = rank_split(y_train, seed=2)

pred_H     = F_test  @ alpha
pred_rbf = kr_cv.predict(X_test)
pred_ridge = ridge_poly.predict(poly.transform(X_test))

print("\n" + "="*70)
print("CLASSIFICATION (seuil = médiane de y_test)")
print("="*70)
for nom, pred in [("Sobolev H", pred_H), ("RBF", pred_rbf), ("Ridge poly", pred_ridge)]:
    # même logique de split par rang pour la prédiction, pour rester cohérent
    # avec y_test_class (sinon on comparerait un split par rang à un split par
    # seuil brut, ce qui n'a pas de sens quand pred sature aussi)
    pred_class = rank_split(pred, seed=1)
    acc = accuracy_score(y_test_class, pred_class)
    try:
        auc = roc_auc_score(y_test_class, pred)
    except Exception:
        auc = float('nan')
    print(f"  {nom:12s} : accuracy={acc:.4f}  AUC={auc:.4f}")


Génération du cloud sur {Q=0}...
  collectés:800/200 (cumulé 100.0%)
  collectés:20000/5000 (cumulé 100.0%)
  collectés:15200/3800 (cumulé 100.0%)
  X_train:(200, 4)  X_unlabeled:(3800, 4)

P_target1 =
(+0.7247)*x2 x3 x4 (+1.4143)*x1 x2^2 (+0.2661)*x1^2 x4 (-0.7296)*x2^2 x3 x4 (+0.9203)*x1^4 (-1.8522)*x1 x2 (-0.3314)*x1 x2^2 x4 (+1.2407)*x1 x3^2 (+0.5739)*x2 x3 x4^2 (+0.8763)*x3^4 (+1.8794)*x2^2 x4 (-0.9866)*x1 x2 x3 (+1.2325)*x1 x2 x4 (+0.3458)*x4 (-1.0467)*x3^3 x4 (+0.2257)*x1 x4^3 (-0.2992)*x2 x4^3 (+0.2570)*x3 x4^2 (-0.8482)*x3 x4^3 (+1.2968)*x2^2 x4^2

P_target2 =
(+0.5308)*x3^2 (-0.8492)*x1^2 x3^2 (-0.6139)*x1 x4^2 (-1.1721)*x2^3 x3 (-0.9148)*x3 x4^3 (-0.9758)*x2^2 x3^2 (+1.2972)*x1^2 x2^2 (-1.6863)*x2 x4^3 (+0.7731)*x4^4 (+0.7163)*x1^2 x2 x4 (+1.2158)*x3 x4^2 (+0.5027)*x2 x3^3 (+1.0416)*x1 x4 (-0.2120)*x1 x2^2 (+0.6894)*x1^3 (-1.2154)*x2 x4 (+0.0052)*x3^2 x4 (-0.2958)*x1^3 x3 (+1.9152)*x1 x2 x3 x4 (-1.1187)*x1 x2 x4^2

Phase 1 : calcul de 1 experts (0 cyl + 25 miso [glouton -> 1

# export

In [16]:
# =============================================================================
# EXPORT EXCEL CUMULATIF — JOUET MISO GLOUTON
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

from openpyxl import Workbook, load_workbook
from pathlib import Path
from datetime import datetime
import os

def excel_value(x):
    """Convertit les types NumPy en types compatibles avec openpyxl."""
    if x is None:
        return None
    if isinstance(x, (float, np.floating)):
        if not np.isfinite(x):
            return None
        return float(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.bool_):
        return bool(x)
    return x


def cls_acc(pred, y_true, seed=1):
    """Accuracy via split par rang."""
    yt = rank_split(y_true, seed=seed)
    yp = rank_split(pred, seed=seed)
    return float(accuracy_score(yt, yp))


def cls_auc(pred, y_true, seed=1):
    yt = rank_split(y_true, seed=seed)
    try:
        return float(roc_auc_score(yt, pred))
    except Exception:
        return float("nan")


def polynomial_to_string(indices, coeffs):
    """Représentation compacte du polynôme."""
    terms = []

    for c, alpha in zip(coeffs, indices):

        if abs(c) < 1e-12:
            continue

        monomial = []

        for j, e in enumerate(alpha):
            if e > 0:
                if e == 1:
                    monomial.append(f"x{j+1}")
                else:
                    monomial.append(f"x{j+1}^{e}")

        if monomial:
            term = " ".join(monomial)
        else:
            term = "1"

        terms.append(f"({c:+.4f})*{term}")

    return " ".join(terms)


# -------------------------------------------------------------------------
# Fichier cumulatif
# -------------------------------------------------------------------------

filepath = Path('/content/drive/MyDrive/jouet_multiexpert_0509.xlsx')

if filepath.exists():
    wb = load_workbook(filepath)
else:
    wb = Workbook()


# =============================================================================
# FEUILLE "Runs"
# =============================================================================

if "Runs" in wb.sheetnames:
    ws = wb["Runs"]
else:
    ws = wb.create_sheet("Runs")

if "Sheet" in wb.sheetnames and len(wb.sheetnames) > 1:
    del wb["Sheet"]


# -------------------------------------------------------------------------
# Numéro du run
# -------------------------------------------------------------------------

if ws.max_row <= 1 and ws["A1"].value is None:
    run_number = 1
else:
    last_run = ws.cell(ws.max_row, 1).value
    try:
        run_number = int(last_run) + 1
    except (TypeError, ValueError):
        run_number = ws.max_row


# -------------------------------------------------------------------------
# En-têtes
# -------------------------------------------------------------------------

headers = [

    "run",

    # Géométrie / données
    "dimension",
    "deg_P",
    "n_terms_poly",
    "n_train",
    "n_unlabeled",
    "n_test",
    "seed_train",

    # MISO
    "sigma_min",
    "sigma_max",
    "lambda_reg",
    "thres_factor",
    "n_levels_miso",
    "n_centres",
    "n_dict",
    "n_G",
    "weights",

    # Cible
    "P_target1",
    "P_target2",

    # Résultats MISO
    "rang_final",
    "MSE_final",
    "MSE_train",
    "norm_H",
    "accuracy_final",
    "AUC_final",

    # RBF
    "RBF_MSE",
    "RBF_accuracy",
    "RBF_AUC",
    "RBF_gamma",
    "RBF_alpha",

    # Ridge polynomial
    "Ridge_MSE",
    "Ridge_accuracy",
    "Ridge_AUC",

    # Meta
    "temps_total",
    "date",
]

if ws.max_row == 1 and ws["A1"].value is None:
    ws.append(headers)


# =============================================================================
# RÉSULTATS FINAUX
# =============================================================================

miso_mse = mean_squared_error(y_test, gr_test)
miso_mse_tr = mean_squared_error(y_train, gr_train)

miso_acc = cls_acc(gr_test, y_test)
miso_auc = cls_auc(gr_test, y_test)

norm_H = (
    float(np.sqrt(gr_h2_hist[-1]))
    if (gr_h2_hist is not None and len(gr_h2_hist))
    else float("nan")
)


# -------------------------------------------------------------------------
# Informations niveaux glouton
# -------------------------------------------------------------------------

_level_info = globals().get("level_info", None)

if not _level_info:
    print(
        "[!] level_info absent — "
        "ré-exécute la cellule principale avant l'export."
    )
    _level_info = []


rang_final = (
    _level_info[-1]["rang"]
    if _level_info
    else None
)

temps_total = (
    float(sum(times_p1))
    if times_p1 is not None
    else float("nan")
)


# -------------------------------------------------------------------------
# Polynômes cibles normalisés
# -------------------------------------------------------------------------

norm1 = np.sqrt(np.mean(coeffst1**2))
norm2 = np.sqrt(np.mean(coeffst2**2))

Ptarget1_string = polynomial_to_string(
    indicest1,
    coeffst1 / norm1
)

Ptarget2_string = polynomial_to_string(
    indicest2,
    coeffst2 / norm2
)


# =============================================================================
# RBF
# =============================================================================

rbf_gamma = kr_cv.best_params_.get("gamma", np.nan)
rbf_alpha = kr_cv.best_params_.get("alpha", np.nan)

pred_rbf = kr_cv.predict(X_test)

rbf_mse = mean_squared_error(y_test, pred_rbf)
rbf_acc = cls_acc(pred_rbf, y_test)
rbf_auc = cls_auc(pred_rbf, y_test)


# =============================================================================
# RIDGE POLYNOMIAL
# =============================================================================

pred_ridge = ridge_poly.predict(poly.transform(X_test))

ridge_mse = mean_squared_error(y_test, pred_ridge)
ridge_acc = cls_acc(pred_ridge, y_test)
ridge_auc = cls_auc(pred_ridge, y_test)


# -------------------------------------------------------------------------
# Accuracy / AUC du dernier niveau
# -------------------------------------------------------------------------

if _level_info:
    _level_info[-1]["acc"] = miso_acc
    _level_info[-1]["auc"] = miso_auc


# =============================================================================
# LIGNE "RUNS"
# =============================================================================

row = [

    run_number,

    # Géométrie / données
    d,
    params.get("deg_P", ""),
    params.get("n_terms_poly", ""),
    params.get("n_train", ""),
    params.get("n_unlabeled", ""),
    params.get("n_test", ""),
    params.get("seeds", {}).get(42, ""),

    # MISO
    params.get("sigma_min", ""),
    params.get("sigma_max", ""),
    params.get("lambda_reg", ""),
    params.get("thres_factor", ""),
    params.get("n_levels_miso", ""),
    params.get("n_centres", ""),
    params.get("n_dict", ""),
    params.get("n_G", ""),
    str(params.get("weights", "")),

    # Cible
    Ptarget1_string,
    Ptarget2_string,

    # Résultats MISO
    rang_final,
    miso_mse,
    miso_mse_tr,
    norm_H,
    miso_acc,
    miso_auc,

    # RBF
    rbf_mse,
    rbf_acc,
    rbf_auc,
    rbf_gamma,
    rbf_alpha,

    # Ridge polynomial
    ridge_mse,
    ridge_acc,
    ridge_auc,

    # Meta
    temps_total,
    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
]

row = [excel_value(x) for x in row]

ws.append(row)


# =============================================================================
# FEUILLE "Glouton"
# =============================================================================

if "Glouton" in wb.sheetnames:
    wg = wb["Glouton"]
else:
    wg = wb.create_sheet("Glouton")


glouton_headers = [
    "run",
    "lev",
    "rang",
    "k",
    "smin",
    "smax",
    "res",
    "mse",
    "acc",
    "auc",
    "h2",
    "loss_res",
    "t",
]

if wg.max_row == 1 and wg["A1"].value is None:
    wg.append(glouton_headers)


for info in _level_info:

    glouton_row = [

        run_number,

        info.get("lev", ""),
        info.get("rang", ""),
        info.get("k", ""),
        info.get("smin", ""),
        info.get("smax", ""),
        info.get("res", ""),
        info.get("mse", ""),
        info.get("acc", ""),
        info.get("auc", ""),
        info.get("h2", ""),
        info.get("loss_res", ""),
        info.get("t", ""),
    ]

    glouton_row = [
        excel_value(x)
        for x in glouton_row
    ]

    wg.append(glouton_row)


# =============================================================================
# MISE EN FORME
# =============================================================================

for sheet in (ws, wg):

    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions

    for column_cells in sheet.columns:

        max_length = 0
        column_letter = column_cells[0].column_letter

        for cell in column_cells:

            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        sheet.column_dimensions[column_letter].width = min(
            max_length + 2,
            40
        )


# Largeur spécifique des polynômes
if "P_target1" in [c.value for c in ws[1]]:
    ws.column_dimensions["T"].width = 60
    ws.column_dimensions["U"].width = 60


# =============================================================================
# SAUVEGARDE
# =============================================================================

wb.save(filepath)


print()
print("=" * 80)
print("RUN AJOUTÉ AU FICHIER CUMULATIF")
print("=" * 80)

print(f"Run       : {run_number}")
print(f"Fichier   : {filepath}")

print(f"MISO MSE  : {miso_mse:.6f}")
print(f"MISO AUC  : {miso_auc:.4f}")
print(f"MISO rang : {rang_final}")

print(
    f"RBF  AUC  : {rbf_auc:.4f}   "
    f"MSE={rbf_mse:.6f}"
)

print(
    f"Ridge AUC : {ridge_auc:.4f}   "
    f"MSE={ridge_mse:.6f}"
)

print(
    f"Niveaux glouton ajoutés : "
    f"{len(_level_info)}"
)

print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[!] level_info absent — ré-exécute la cellule principale avant l'export.

RUN AJOUTÉ AU FICHIER CUMULATIF
Run       : 4
Fichier   : /content/drive/MyDrive/jouet_multiexpert_0509.xlsx
MISO MSE  : 0.010838
MISO AUC  : 0.9951
MISO rang : None
RBF  AUC  : 0.9918   MSE=0.015196
Ridge AUC : 0.9691   MSE=0.427243
Niveaux glouton ajoutés : 0


#fin

